# NLP Preprocessing Engine
**Data Science Internship – February 2026**  
A robust, modular NLP preprocessing pipeline for real-world noisy text.

---
## Task 1: Conceptual Understanding

**Q1. What is the difference between "Love" and "love" in NLP?**  
In NLP, "Love" and "love" are treated as different tokens unless text is lowercased. Case sensitivity matters in certain contexts — for example, "Love" at the start of a sentence might be a proper name or an intentional emphasis. In most ML pipelines, we convert to lowercase so the model treats both as the same word and doesn't split their frequency/meaning artificially.

**Q2. What happens if stopwords are not removed?**  
Stopwords (e.g., "the", "is", "and") are extremely frequent but carry little semantic meaning. If not removed, they dominate frequency counts and vector representations, drowning out the meaningful words. This leads to poor model performance — the model wastes capacity learning useless patterns.

**Q3. Two real-world scenarios where removing stopwords can be harmful:**  
1. **Sentiment analysis with negation**: "I am *not* happy" — removing "not" completely flips the sentiment from negative to positive. The word "not" is a stopword but is semantically critical.  
2. **Question answering / information retrieval**: "To be or not to be" — every word here is a stopword, yet the phrase has strong cultural meaning and specific retrieval intent. Removing stopwords would make this query unrecognizable.

**Q4. Difference between stemming and lemmatization:**  
- **Stemming** is a crude, rule-based process that chops word endings to find a rough root form. It's fast but inaccurate — e.g., "running" → "run", but "studies" → "studi" (not a real word).  
- **Lemmatization** uses vocabulary and morphological analysis to return the actual dictionary base form (the *lemma*). It's slower but accurate — e.g., "better" → "good", "studies" → "study". Lemmatization is preferred when word meaning matters.

---
## Setup & Imports

In [2]:
import re
import string
from collections import Counter

# Install NLTK if needed and download resources
try:
    import nltk
    nltk.download('stopwords', quiet=True)
    nltk.download('punkt', quiet=True)
    from nltk.corpus import stopwords
    STOPWORDS = set(stopwords.words('english'))
except ImportError:
    print("NLTK not available — using a basic stopword list.")
    STOPWORDS = {
        'i','me','my','myself','we','our','ours','ourselves','you','your','yours',
        'yourself','yourselves','he','him','his','himself','she','her','hers',
        'herself','it','its','itself','they','them','their','theirs','themselves',
        'what','which','who','whom','this','that','these','those','am','is','are',
        'was','were','be','been','being','have','has','had','having','do','does',
        'did','doing','a','an','the','and','but','if','or','because','as','until',
        'while','of','at','by','for','with','about','against','between','into',
        'through','during','before','after','above','below','to','from','up','down',
        'in','out','on','off','over','under','again','further','then','once',
        'here','there','when','where','why','how','all','both','each','few','more',
        'most','other','some','such','only','own','same','so','than','too','very',
        's','t','can','will','just','don','should','now','d','ll','m','o','re',
        've','y','ain','aren','couldn','didn','doesn','hadn','hasn','haven',
        'isn','ma','mightn','mustn','needn','shan','shouldn','wasn','weren',
        'won','wouldn'
    }

# Words ≤ 2 chars to keep even though they are short
KEEP_SHORT = {'no', 'not', 'ok', 'oh', 'go', 'do', 'is', 'am', 'be'}

print("✅ Libraries loaded successfully.")

✅ Libraries loaded successfully.


---
## Task 2: Advanced Preprocessing Function

In [3]:
def preprocess_text(text):
    """
    Advanced NLP preprocessing pipeline.
    
    Steps:
      1. Remove URLs and email-like patterns
      2. Remove emojis and non-ASCII characters
      3. Convert to lowercase
      4. Handle repeated characters (e.g., 'soooo' → 'so')
      5. Remove punctuation and special characters
      6. Remove numbers
      7. Tokenize
      8. Remove stopwords (keeping negations)
      9. Remove very short tokens (length ≤ 2), except meaningful ones
     10. Remove extra whitespace
    
    Returns:
        tokens (list): cleaned token list
        clean_sentence (str): tokens joined as a sentence
    """
    if not isinstance(text, str) or not text.strip():
        return [], ""
    
    # --- Step 1: Remove URLs (http/https/www) and email-like patterns ---
    text = re.sub(r'http\S+|www\.\S+', '', text)       # URLs
    text = re.sub(r'\S+@\S+\.\S+', '', text)            # Emails
    
    # --- Step 2: Remove emojis and non-ASCII ---
    text = text.encode('ascii', 'ignore').decode('ascii')
    
    # --- Step 3: Lowercase ---
    text = text.lower()
    
    # --- Step 4: Handle repeated characters (3+ repetitions → 1) ---
    # e.g. "looooove" → "love", "!!!!!" → "!"
    text = re.sub(r'(.)\1{2,}', r'\1', text)
    
    # --- Step 5: Remove punctuation and special characters ---
    text = re.sub(r'[^\w\s]', ' ', text)   # keep word chars and spaces
    text = re.sub(r'_', ' ', text)          # underscores too
    
    # --- Step 6: Remove numbers (standalone digits) ---
    text = re.sub(r'\b\d+\b', '', text)
    
    # --- Step 7: Tokenize ---
    tokens = text.split()
    
    # --- Step 8: Remove stopwords, but keep meaningful negation words ---
    # We skip this step to preserve 'not', 'no' which may be in stopwords
    # Instead, we only remove stopwords that are not in KEEP_SHORT
    tokens = [
        t for t in tokens
        if t not in STOPWORDS or t in KEEP_SHORT
    ]
    
    # --- Step 9: Remove very short tokens (length ≤ 2) unless meaningful ---
    tokens = [t for t in tokens if len(t) > 2 or t in KEEP_SHORT]
    
    # --- Step 10: Remove empty tokens ---
    tokens = [t for t in tokens if t]
    
    clean_sentence = ' '.join(tokens)
    return tokens, clean_sentence

print("✅ preprocess_text() defined.")

# Quick sanity checks
print(preprocess_text("I have 2 dogs"))
print(preprocess_text("This  is   good"))
print(preprocess_text("soooo goooood!!!"))
print(preprocess_text("WOW!!! This is GREAT!!!"))
print(preprocess_text("Visit http://example.com now"))

✅ preprocess_text() defined.
(['dogs'], 'dogs')
(['is', 'good'], 'is good')
(['god'], 'god')
(['wow', 'is', 'great'], 'wow is great')
(['visit'], 'visit')


---
## Task 3: Stress Testing on 10 Diverse Sentences

In [4]:
test_sentences = [
    "Get 100% FREE access now!!!",
    "I absolutely looooved this product 😍😍",
    "Worst service ever... 0/10",
    "Call me at 9876543210",
    "This is THE best course!!!",
    "Visit https://openai.com now!",
    "Nooooo this is baaad!!!",
    "OK OK OK I got it",
    "Win $$$ now!!! Limited offer!!!",
    "I am not happy with this"
]

results = []

print("=" * 70)
for i, sentence in enumerate(test_sentences, 1):
    tokens, clean = preprocess_text(sentence)
    results.append({'original': sentence, 'tokens': tokens, 'clean': clean})
    print(f"\n[Sentence {i}]")
    print(f"  Original  : {sentence}")
    print(f"  Tokens    : {tokens}")
    print(f"  Cleaned   : {clean}")
print("\n" + "=" * 70)


[Sentence 1]
  Original  : Get 100% FREE access now!!!
  Tokens    : ['get', 'free', 'access']
  Cleaned   : get free access

[Sentence 2]
  Original  : I absolutely looooved this product 😍😍
  Tokens    : ['absolutely', 'loved', 'product']
  Cleaned   : absolutely loved product

[Sentence 3]
  Original  : Worst service ever... 0/10
  Tokens    : ['worst', 'service', 'ever']
  Cleaned   : worst service ever

[Sentence 4]
  Original  : Call me at 9876543210
  Tokens    : ['call']
  Cleaned   : call

[Sentence 5]
  Original  : This is THE best course!!!
  Tokens    : ['is', 'best', 'course']
  Cleaned   : is best course

[Sentence 6]
  Original  : Visit https://openai.com now!
  Tokens    : ['visit']
  Cleaned   : visit

[Sentence 7]
  Original  : Nooooo this is baaad!!!
  Tokens    : ['no', 'is', 'bad']
  Cleaned   : no is bad

[Sentence 8]
  Original  : OK OK OK I got it
  Tokens    : ['ok', 'ok', 'ok', 'got']
  Cleaned   : ok ok ok got

[Sentence 9]
  Original  : Win $$$ now!!! Limite

---
## Task 4: Token Analytics

In [5]:
print(f"{'#':<4} {'Total':>7} {'Unique':>7} {'Avg Len':>8}  Sentence Preview")
print("-" * 70)

analytics = []
for i, r in enumerate(results, 1):
    tokens = r['tokens']
    total = len(tokens)
    unique = len(set(tokens))
    avg_len = round(sum(len(t) for t in tokens) / total, 2) if total > 0 else 0
    analytics.append({'total': total, 'unique': unique, 'avg_len': avg_len})
    preview = r['original'][:40] + ('...' if len(r['original']) > 40 else '')
    print(f"{i:<4} {total:>7} {unique:>7} {avg_len:>8}  {preview}")

# --- Analysis Questions ---
print("\n" + "=" * 70)

# Sentence with the most noise = fewest tokens remaining after cleaning
most_noise_idx = min(range(len(analytics)), key=lambda i: analytics[i]['total'])
print(f"\n📌 Most noisy sentence (fewest tokens after cleaning):")
print(f"   Sentence {most_noise_idx+1}: '{results[most_noise_idx]['original']}'")
print(f"   Tokens remaining: {analytics[most_noise_idx]['total']}")

# Sentence retaining the most meaningful tokens
most_meaningful_idx = max(range(len(analytics)), key=lambda i: analytics[i]['total'])
print(f"\n📌 Most meaningful sentence (most tokens retained):")
print(f"   Sentence {most_meaningful_idx+1}: '{results[most_meaningful_idx]['original']}'")
print(f"   Tokens retained: {analytics[most_meaningful_idx]['total']}")

#      Total  Unique  Avg Len  Sentence Preview
----------------------------------------------------------------------
1          3       3     4.33  Get 100% FREE access now!!!
2          3       3     7.33  I absolutely looooved this product 😍😍
3          3       3     5.33  Worst service ever... 0/10
4          1       1      4.0  Call me at 9876543210
5          3       3      4.0  This is THE best course!!!
6          1       1      5.0  Visit https://openai.com now!
7          3       3     2.33  Nooooo this is baaad!!!
8          4       2     2.25  OK OK OK I got it
9          3       3      5.0  Win $$$ now!!! Limited offer!!!
10         3       3     3.33  I am not happy with this


📌 Most noisy sentence (fewest tokens after cleaning):
   Sentence 4: 'Call me at 9876543210'
   Tokens remaining: 1

📌 Most meaningful sentence (most tokens retained):
   Sentence 8: 'OK OK OK I got it'
   Tokens retained: 4


---
## Task 5: Frequency Analysis

In [6]:
from collections import Counter

# Combine all tokens from all sentences
all_tokens = []
for r in results:
    all_tokens.extend(r['tokens'])

token_counts = Counter(all_tokens)

print("📊 Top 10 Most Frequent Words:")
print("-" * 30)
for word, count in token_counts.most_common(10):
    bar = '█' * count
    print(f"  {word:<15} {count:>3}  {bar}")

print("\n📊 Top 5 Least Frequent Words:")
print("-" * 30)
least_common = token_counts.most_common()[:-6:-1]  # last 5
for word, count in least_common:
    bar = '░' * count
    print(f"  {word:<15} {count:>3}  {bar}")

📊 Top 10 Most Frequent Words:
------------------------------
  ok                3  ███
  is                2  ██
  get               1  █
  free              1  █
  access            1  █
  absolutely        1  █
  loved             1  █
  product           1  █
  worst             1  █
  service           1  █

📊 Top 5 Least Frequent Words:
------------------------------
  happy             1  ░
  not               1  ░
  am                1  ░
  offer             1  ░
  limited           1  ░


---
## Task 6: Full Pipeline

In [7]:
def full_pipeline(text_list):
    """
    Full NLP preprocessing pipeline for a list of text strings.
    
    Args:
        text_list (list): List of raw text strings.
    
    Returns:
        dict: {
            'tokens': list of token lists (one per sentence),
            'clean_sentences': list of cleaned sentence strings
        }
    """
    if not isinstance(text_list, list):
        raise TypeError("Input must be a list of strings.")
    
    tokens_all = []
    clean_sentences = []
    
    for text in text_list:
        tokens, clean = preprocess_text(text)
        tokens_all.append(tokens)
        clean_sentences.append(clean)
    
    return {
        "tokens": tokens_all,
        "clean_sentences": clean_sentences
    }

# Demo
output = full_pipeline(test_sentences)
print("full_pipeline() output keys:", list(output.keys()))
print("\nSample clean_sentences:")
for i, s in enumerate(output['clean_sentences'], 1):
    print(f"  {i}. {s}")

full_pipeline() output keys: ['tokens', 'clean_sentences']

Sample clean_sentences:
  1. get free access
  2. absolutely loved product
  3. worst service ever
  4. call
  5. is best course
  6. visit
  7. no is bad
  8. ok ok ok got
  9. win limited offer
  10. am not happy


---
## Task 7: Error Handling

In [8]:
edge_cases = [
    "",                    # Empty string
    "😍🔥💥🎉😂",           # Only emojis
    "1234567890",          # Only numbers
    "   ",                 # Only spaces
    "!!!! @@@ ###",        # Only punctuation
]

print("Edge Case Handling\n" + "=" * 50)
for case in edge_cases:
    tokens, clean = preprocess_text(case)
    display = repr(case) if len(case) < 30 else repr(case[:30]) + '...'
    print(f"  Input   : {display}")
    print(f"  Tokens  : {tokens}")
    print(f"  Cleaned : '{clean}'")
    print()

# Also test full_pipeline with type error
try:
    full_pipeline("not a list")
except TypeError as e:
    print(f"TypeError caught correctly: {e}")

Edge Case Handling
  Input   : ''
  Tokens  : []
  Cleaned : ''

  Input   : '😍🔥💥🎉😂'
  Tokens  : []
  Cleaned : ''

  Input   : '1234567890'
  Tokens  : []
  Cleaned : ''

  Input   : '   '
  Tokens  : []
  Cleaned : ''

  Input   : '!!!! @@@ ###'
  Tokens  : []
  Cleaned : ''

TypeError caught correctly: Input must be a list of strings.


---
## Summary

| Task | Description | Status |
|------|-------------|--------|
| Task 1 | Conceptual questions answered | ✅ |
| Task 2 | `preprocess_text()` with all required features | ✅ |
| Task 3 | Stress tested on 10 diverse sentences | ✅ |
| Task 4 | Token analytics (total, unique, avg length) | ✅ |
| Task 5 | Frequency analysis with Counter | ✅ |
| Task 6 | `full_pipeline()` returning dict output | ✅ |
| Task 7 | Edge case error handling | ✅ |